**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Kernel Methods & RKHS

The elegant middle path between linear models and neural networks — and a proud UF lineage (information-theoretic learning and kernel adaptive filtering grew up here). Kernel trick, Gaussian processes with honest error bars, and KLMS: the [adaptive filter](../Intro_Time_Series/Intro_AdFilt_APA.ipynb) gone nonlinear.

## 1. Pre-requisites

- [Hilbert Spaces](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb) — inner products, projections (an RKHS is a Hilbert space with a bonus property).
- [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S2, [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb) S3 for the GP session.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

def rbf(A, B, ell=0.5):
    """the Gaussian (RBF) kernel — similarity that decays with distance"""
    d2 = ((A[:, None, :] - B[None, :, :])**2).sum(-1)
    return np.exp(-d2 / (2 * ell**2))

---
### 🕐 Session 1 of 3 — *The Kernel Trick* (~35 min)
**Goal:** replace inner products with kernels; fit nonlinear functions with linear algebra.
**Builds on:** [Hilbert Spaces](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb). &nbsp; **Feeds into:** Session 2 (Gaussian processes).

---

## 2. Features Without Features

💡 **Intuition.** Linear methods only see inner products $x_i^T x_j$. The trick: replace every inner product with a **kernel** $k(x_i, x_j)$ — a similarity function that secretly equals an inner product in some (possibly infinite-dimensional) feature space. You get nonlinear power at linear-algebra prices, without ever visiting the feature space. The **representer theorem** seals it: the optimal function is always a weighted sum of kernels *centered on your data* — $f(x) = \sum_i \alpha_i k(x_i, x)$ — so the infinite-dimensional search collapses to solving for $n$ numbers. An RKHS is the [Hilbert space](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb) where evaluation *is* an inner product with a kernel bump.

In [2]:
# Kernel ridge regression from scratch: (K + λI)α = y — one linear solve, nonlinear fit
x = rng.uniform(-3, 3, 60)[:, None]
y = np.sin(2*x[:, 0]) + 0.5*np.sign(x[:, 0]) + 0.15*rng.standard_normal(60)   # kinked + noisy

lamr = 0.1
K = rbf(x, x)
alpha = np.linalg.solve(K + lamr*np.eye(len(x)), y)

xs = np.linspace(-3.4, 3.4, 400)[:, None]
f_hat = rbf(xs, x) @ alpha

plt.figure(figsize=(8, 2.8))
plt.plot(x, y, "k.", markersize=5, label="data")
plt.plot(xs, f_hat, label="kernel ridge (60 α's, one solve)")
plt.plot(xs, np.sin(2*xs) + 0.5*np.sign(xs), "k--", linewidth=0.8, label="truth")
plt.legend(fontsize=8); plt.title("nonlinear regression with nothing but linear algebra + a kernel")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2696075/3499237204.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**The lengthscale is the model.** $\ell$ controls how far influence spreads — small $\ell$ wiggles (overfits), large $\ell$ oversmooths. It's the bias-variance dial in one number, and choosing it honestly (cross-validation, or S2's marginal likelihood) is most of kernel practice.

---
### 🕐 Session 2 of 3 — *Gaussian Processes* (~40 min)
**Goal:** put a prior on functions; get predictions WITH calibrated uncertainty.
**Builds on:** Session 1; [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb) S3. &nbsp; **Feeds into:** Session 3 (kernel adaptive filters).

---

## 3. Distributions Over Functions

💡 **Intuition.** A GP is [Bayesian estimation](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb) upgraded from parameters to *whole functions*: the kernel plays the prior ('smooth functions with lengthscale ℓ are likely'), data updates it, and the posterior at any test point is a Gaussian — mean **and variance**, in closed form. The error bars behave the way honesty demands: pinched near data, ballooning in the gaps. Where a neural net extrapolates with confidence it hasn't earned, a GP *tells you it doesn't know*.

In [3]:
sn = 0.15                                        # known noise level
Kxx = rbf(x, x) + sn**2*np.eye(len(x))
Ks  = rbf(xs, x)
Kss = rbf(xs, xs)

L = np.linalg.cholesky(Kxx)
alpha_gp = np.linalg.solve(L.T, np.linalg.solve(L, y))
mu = Ks @ alpha_gp
v = np.linalg.solve(L, Ks.T)
var = np.diag(Kss) - (v**2).sum(0)
sd = np.sqrt(np.maximum(var, 0))

plt.figure(figsize=(8, 2.8))
plt.plot(x, y, "k.", markersize=5)
plt.plot(xs, mu, label="GP posterior mean")
plt.fill_between(xs[:, 0], mu-2*sd, mu+2*sd, alpha=0.2, label="±2σ")
plt.legend(fontsize=8); plt.title("uncertainty pinches at data, balloons in the gaps — as it should")
plt.tight_layout(); plt.show()

# calibration audit on fresh data
x_new = rng.uniform(-3, 3, 500)[:, None]
y_new = np.sin(2*x_new[:, 0]) + 0.5*np.sign(x_new[:, 0]) + 0.15*rng.standard_normal(500)
mu_new = rbf(x_new, x) @ alpha_gp
v2 = np.linalg.solve(L, rbf(x, x_new))
sd_new = np.sqrt(np.maximum(1 - (v2**2).sum(0) + sn**2, 0))
inside = np.mean(np.abs(y_new - mu_new) <= 2*sd_new)
print(f"fresh points inside the ±2σ band: {inside:.0%}  (well-calibrated ≈ 95%)")

fresh points inside the ±2σ band: 92%  (well-calibrated ≈ 95%)


/tmp/ipykernel_2696075/1619739755.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 3 of 3 — *Kernel Adaptive Filters: KLMS* (~40 min)
**Goal:** run LMS in the RKHS: a nonlinear adaptive filter, one sample at a time.
**Builds on:** Session 1; [APA workshop](../Intro_Time_Series/Intro_AdFilt_APA.ipynb).

---

## 4. LMS Meets the Kernel

💡 **Intuition.** [LMS](../Intro_Time_Series/Intro_AdFilt_APA.ipynb) updates a weight vector; **KLMS** runs the identical update *in the RKHS* — by the representer theorem the filter is a growing sum of kernel bumps, one planted on each sample, weighted by $\mu e_n$: $f_n = f_{n-1} + \mu e_n \, k(x_n, \cdot)$. It learns *nonlinear* systems online with LMS's simplicity. The price is the growing dictionary — practical variants (QKLMS) merge nearby bumps to cap it.

In [4]:
# nonlinear system id: y = tanh of a filtered input — LMS can't, KLMS can
from scipy import signal as sig
N = 2000
u = rng.standard_normal(N)
lin = sig.lfilter([1, 0.5, -0.3], [1], u)
d = np.tanh(1.5*lin) + 0.05*rng.standard_normal(N)

L_emb = 5                                          # embed the last 5 inputs as the "x"
U = np.stack([np.roll(u, k) for k in range(L_emb)], 1); U[:L_emb] = 0

def lms_lin(U, d, mu=0.05):
    w = np.zeros(U.shape[1]); e = np.zeros(len(d))
    for t in range(len(d)):
        e[t] = d[t] - w @ U[t]
        w += mu * e[t] * U[t]
    return e

def klms(U, d, mu=0.5, ell=1.0):
    centers, coefs = [], []
    e = np.zeros(len(d))
    for t in range(len(d)):
        if centers:
            kv = np.exp(-((np.array(centers) - U[t])**2).sum(1) / (2*ell**2))
            y = np.dot(coefs, kv)
        else:
            y = 0.0
        e[t] = d[t] - y
        centers.append(U[t].copy()); coefs.append(mu * e[t])
    return e

e_lin, e_k = lms_lin(U, d), klms(U, d)
def curve(e): return 10*np.log10(np.convolve(e**2, np.ones(80)/80, "valid") + 1e-12)
plt.figure(figsize=(8, 2.8))
plt.plot(curve(e_lin), label="linear LMS: floored by the nonlinearity")
plt.plot(curve(e_k), label="KLMS: keeps descending")
plt.legend(fontsize=8); plt.grid(True, alpha=0.3); plt.xlabel("sample"); plt.ylabel("MSE [dB]")
plt.title("nonlinear system: the kernel buys what no linear width can")
plt.tight_layout(); plt.show()
print(f"steady-state MSE  linear {np.mean(e_lin[-500:]**2):.4f}   KLMS {np.mean(e_k[-500:]**2):.4f}")

steady-state MSE  linear 0.0931   KLMS 0.0288


/tmp/ipykernel_2696075/2130777877.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 5. Conclusion

Kernels buy nonlinearity at linear-algebra prices; GPs add honest uncertainty; KLMS carries it all online. When data is scarce and error bars matter, this toolbox still beats deep learning — and when data is huge, you'll understand *why* networks won (the dictionary problem).

---
## Where next

- [Uncertainty in ML](./Uncertainty_in_ML.ipynb) — chasing GP-quality error bars with deep models.
- [RLS workshop](../Intro_Time_Series/Intro_RLS.ipynb) — KRLS: the recursive version.
- [Hilbert Spaces](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb) — the geometry under all of it.